# SharePoint recycle bin purge: how to use it

This notebook walks an administrator through `sharepoint_recyclebin_purge.py`, the script that clears a SharePoint site's recycle bin so deleted files stop counting against the storage quota.

> **A purge is permanent.** The recycle bin is shared by everyone on the site. Anything the script deletes cannot be restored by its owner. That is why it **never deletes unless you pass `--execute`**, only touches items older than 30 days by default, refuses to run on more than 10,000 items, and writes a CSV log of everything it selects.

**The safe order of work:** configure, dry run, look at the log, narrow the filters, execute, check again.

Keep this notebook next to `sharepoint_recyclebin_purge.py`.

## How the SharePoint recycle bin works

| Stage | What it holds | Counts toward quota? |
|---|---|---|
| First stage | Files just deleted by users. They can restore them. | Yes |
| Second stage | Files users emptied from their bin, or admins deleted. Only admins can restore them. | Yes |

Items leave the second stage on their own about 93 days after deletion. The script's `--stage` option picks which stage to select:

- `all` (default): purge both stages. If a first-stage item only moves to the second stage when deleted, the script deletes it again so it is really gone. It only re-deletes items it selected itself.
- `second`: purge only the second stage.
- `first`: only moves items to the second stage. This frees **no** quota.

The Microsoft Graph API used by the IM pipeline cannot do this for a regular site, so the script uses the SharePoint REST API (`_api/site/RecycleBin`).

## Step 0: One-time prerequisites (IT)

1. **Create an app registration in Entra ID that signs in with a certificate.** SharePoint REST refuses client-secret app-only tokens. Upload the certificate's public part to the app.
2. **Give the app a SharePoint permission and grant admin consent:** `Sites.FullControl.All`, or `Sites.Selected` with *full control* granted on this one site only (preferred, it limits the blast radius).
3. **Export the certificate's private key as a PEM file:**

```
openssl pkcs12 -in cert.pfx -nocerts -nodes -out key.pem
```

4. Note the **tenant ID**, the app's **client ID** and the certificate **thumbprint** (hex, shown in Entra ID).
5. Keep `key.pem` out of git and off shared drives.

In [ ]:
# Install the two packages the script needs (pandas is only used further down to read the log).
%pip install -q msal requests pandas

## Step 1: Configure

Fill in the five values below. The script reads them from environment variables, so for unattended runs you would set the same variables in the scheduler instead of in a notebook. If the key file is password-protected, also set `SP_CERT_PASSPHRASE`.

In [ ]:
import os, sys, glob, subprocess
from pathlib import Path

CONFIG = {
    "SP_SITE_URL":        "<https://TENANT.sharepoint.com/sites/AF-pep/GISWORKSPACE>",
    "SP_TENANT_ID":       "<directory (tenant) id>",
    "SP_APP_CLIENT_ID":   "<application (client) id>",
    "SP_CERT_KEY_PATH":   "<C:/path/to/key.pem>",
    "SP_CERT_THUMBPRINT": "<certificate thumbprint>",
}
# os.environ["SP_CERT_PASSPHRASE"] = "..."   # only if the PEM key is encrypted

os.environ.update(CONFIG)
unfilled = [k for k, v in CONFIG.items() if v.startswith("<")]
if unfilled:
    print("Still to fill in:", ", ".join(unfilled))
else:
    print("Configuration looks complete.")

In [ ]:
# Helper used by every step below: runs the script and shows its output.
def find_script(name="sharepoint_recyclebin_purge.py"):
    for folder in (Path.cwd(), Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(f"Put {name} next to this notebook.")

SCRIPT = find_script()
EXIT_MEANING = {
    0: "OK (a dry run always ends here)",
    1: "Finished, but some deletions failed (see the log)",
    2: "Configuration, permission or sign-in problem",
    3: "Safety stop: too many items, or the confirmation was not given",
}

def purge(*args):
    if unfilled:
        print("Fill in the configuration cell first:", ", ".join(unfilled))
        return None
    proc = subprocess.run([sys.executable, str(SCRIPT), *args],
                          capture_output=True, text=True)
    print(proc.stdout, end="")
    if proc.stderr:
        print(proc.stderr, end="")
    print(f"\n[exit code {proc.returncode}: {EXIT_MEANING.get(proc.returncode, 'unknown')}]")
    return proc.returncode

print("Script:", SCRIPT)

## Step 2: Dry run (changes nothing)

With no flags the script signs in, reads the whole site-collection recycle bin, applies the default filters (items deleted more than 30 days ago, both stages) and prints:

- the total size of the bin,
- how many items and how much space the filters selected,
- the split by stage and by **who deleted** the items,
- the first 15 items, with their original paths.

It also writes a CSV to `recyclebin_logs/`. **Nothing is deleted.**

In [ ]:
purge()

## Step 3: Narrow the filters

Run the dry run again with filters until the selection is exactly what you intend to remove. All filters can be combined.

| Flag | Meaning |
|---|---|
| `--older-than-days N` | Only items deleted more than N days ago (default 30, `0` means everything) |
| `--path-contains TEXT` | Only items whose original path contains TEXT (repeatable) |
| `--exclude-path TEXT` | Never touch items whose original path contains TEXT (repeatable) |
| `--stage first\|second\|all` | Which stage to select (default `all`) |
| `--max-items N` | Refuse to run if more than N items are selected (default 10000) |

In [ ]:
# Example: only the IM pipeline's raw_state folder, older than 60 days.
purge("--path-contains", "Cloud-Independant-Monitoring/raw_state", "--older-than-days", "60")

# More examples (uncomment one to try it):
# purge("--older-than-days", "90")                                   # only very old items
# purge("--exclude-path", "Weekly Pager", "--exclude-path", "Virus Tracking")   # protect folders
# purge("--stage", "second", "--older-than-days", "0")               # everything in the second stage

## Step 4: Read the log before deciding

Every run writes `recyclebin_logs/recyclebin_dryrun_<timestamp>.csv` (or `..._execute_...` for a real run) with one row per item: when, what, which stage, deletion date, size, who deleted it, and the original path. Review it, or send it to the site owners if the selection includes other people's files.

In [ ]:
import pandas as pd

logs = sorted(Path("recyclebin_logs").glob("recyclebin_dryrun_*.csv"))
if not logs:
    print("No dry-run log yet. Run Step 2 first.")
else:
    df = pd.read_csv(logs[-1])
    print("Latest dry-run log:", logs[-1].name, "->", len(df), "items")
    df["size_mb"] = df["size_bytes"] / 1024 / 1024
    df["folder"] = df["path"].str.rsplit("/", n=1).str[0].str.split("/").str[:6].str.join("/")
    display(df.groupby("deleted_by")["size_mb"].agg(["count", "sum"]).sort_values("sum", ascending=False).round(2).head(10))
    display(df.groupby("folder")["size_mb"].agg(["count", "sum"]).sort_values("sum", ascending=False).round(2).head(10))
    display(df.sort_values("size_mb", ascending=False)[["deleted_by", "deleted_utc", "size_mb", "path"]].head(10))

## Step 5: Execute (permanent)

Only do this after a dry run whose log you have reviewed, using **the same filters**. Set `EXECUTE = True`, run the cell once, then set it back to `False`.

Running the script from a terminal instead asks you to type `DELETE <number of items>` before it proceeds. In a notebook the script cannot ask, so this cell uses `--yes` and refuses to run unless a dry-run log already exists.

In [ ]:
EXECUTE = False   # <-- set to True deliberately, run this cell once, then set back to False

# Use exactly the filters you tested in Step 3:
EXECUTE_ARGS = ["--older-than-days", "30"]

if not EXECUTE:
    print("EXECUTE is False: nothing will be deleted.")
elif not list(Path("recyclebin_logs").glob("recyclebin_dryrun_*.csv")):
    print("No dry-run log found. Run Step 2 first.")
else:
    purge("--execute", "--yes", *EXECUTE_ARGS)

## Step 6: Check the result

The execute run prints how many delete calls succeeded and failed and how many of the selected items are still visible, and it writes the audit log (`recyclebin_execute_...csv`). A dry run with the same filters afterwards should say **Nothing to do**. Quota figures in SharePoint admin reports can take several hours to update.

In [ ]:
purge("--older-than-days", "30")

## Running it on a schedule (optional)

Only schedule it after several dry runs looked right. A scheduled run has no one to confirm, so it uses `--yes`. Keep a conservative age limit and a lower `--max-items` as a safety net.

**Windows Task Scheduler**: action: `python`, arguments:

```
C:\tools\sharepoint_recyclebin_purge.py --execute --yes --older-than-days 60 --max-items 2000 --log-dir C:\tools\recyclebin_logs
```

Set the five `SP_...` variables for the account that runs the task. Check the log folder regularly.

**Linux cron** (Sundays at 02:00):

```
0 2 * * 0  /usr/bin/python3 /opt/tools/sharepoint_recyclebin_purge.py --execute --yes --older-than-days 60 --max-items 2000 --log-dir /var/log/recyclebin
```

If a run stops on the `--max-items` safety limit (exit code 3), that is the safety net doing its job. Look at the log before raising the limit.

## Troubleshooting

| What you see | Likely cause and fix |
|---|---|
| `Missing configuration: ...` | One of the five `SP_...` values is empty. Fill it in. |
| `Missing Python package` | Run `pip install msal requests`. |
| `ABORTED: Token request failed` | Wrong tenant/client ID, thumbprint, or the certificate is not uploaded to the app. |
| `ABORTED: HTTP 401/403 from SharePoint` | The app lacks `Sites.FullControl.All` (or `Sites.Selected` with full control on this site), or admin consent is missing. A client secret will not work, it must be a certificate. |
| `REFUSING: N items selected, above --max-items` | The safety cap. Narrow the filters, or raise `--max-items` on purpose. |
| `Confirmation did not match` | The phrase typed was not exactly `DELETE <N>`. Nothing was deleted. |
| Some items "failed" in the log | Read the `result` column. Failed items are left in place. Re-run to retry. |
| Quota did not drop right away | Quota reports lag by hours. Also check **Site settings, Storage Metrics**: version history of live files is often the bigger consumer than the bin. |

**Exit codes:** 0 ok or dry run, 1 some deletions failed, 2 configuration or authentication error, 3 safety limit hit or confirmation declined.